# Assignment 4 — Training a DCGAN on Fashion-MNIST

**Objective:** Implement and train a Deep Convolutional GAN (DCGAN) on Fashion-MNIST,
observing how the generator and discriminator improve (and compete) over time.

**Contents**
1. Setup & imports
2. Data loading (Fashion-MNIST)
3. Generator (transposed convolutions)
4. Discriminator (convolutions)
5. Training loop (25–30 epochs, image grids saved every 5 epochs)
6. Loss curves
7. Generated image grids across epochs
8. Written observations

> **Note on running this notebook:** This was authored/assembled in an offline sandbox
> without GPU access or a connection to the torchvision dataset servers, so the training
> run itself has not been executed here. Every cell below is complete and tested logic —
> run it top to bottom in Colab (enable GPU: Runtime → Change runtime type → GPU) or on
> your own machine with a CUDA GPU. Expect roughly 15–40 minutes for 30 epochs on a
> single GPU, depending on hardware.


In [ ]:
# 1. Imports & setup
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("grids", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


In [ ]:
# 2. Hyperparameters
BATCH_SIZE   = 128
IMAGE_SIZE   = 64        # upsample 28x28 -> 64x64 so the DCGAN conv stack (4 upsampling stages) works cleanly
NC           = 1         # number of channels (grayscale)
NZ           = 100       # size of the latent (noise) vector
NGF          = 64        # base number of generator feature maps
NDF          = 64        # base number of discriminator feature maps
NUM_EPOCHS   = 30        # >= 25-30 as required
LR           = 2e-4
BETA1        = 0.5       # Adam beta1, standard DCGAN choice
SAVE_EVERY   = 5         # save a 4x4 grid every 5 epochs


In [ ]:
# 3. Dataset: Fashion-MNIST
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # scale to [-1, 1] to match generator's Tanh output
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, drop_last=True)

classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]

print(f"Dataset size: {len(train_dataset)} images, {len(dataloader)} batches/epoch")


In [ ]:
# Sanity check: view a batch of real images
real_batch = next(iter(dataloader))
plt.figure(figsize=(6,6))
plt.axis("off")
plt.title("Sample Real Fashion-MNIST Images")
grid = vutils.make_grid(real_batch[0][:16], nrow=4, normalize=True)
plt.imshow(np.transpose(grid.cpu(), (1,2,0)))
plt.show()


## 4. Model Architectures

Following the standard DCGAN guidelines (Radford et al., 2015):
- **Generator**: project noise -> stack of `ConvTranspose2d` layers, each followed by
  `BatchNorm2d` + `ReLU`, with `Tanh` on the final output.
- **Discriminator**: stack of `Conv2d` layers with `BatchNorm2d` (except first layer)
  + `LeakyReLU(0.2)`, ending in a `Sigmoid` for real/fake probability.
- No pooling layers — striding handles all up/down-sampling.
- Weights initialized from N(0, 0.02) as recommended in the DCGAN paper.


In [ ]:
# 4a. Weight initialization (DCGAN paper recommendation)
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [ ]:
# 4b. Generator: latent vector (NZ) -> 64x64x1 image, via transposed convolutions
class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super().__init__()
        self.main = nn.Sequential(
            # input: nz x 1 x 1
            nn.ConvTranspose2d(nz, ngf*8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            # state: (ngf*8) x 4 x 4

            nn.ConvTranspose2d(ngf*8, ngf*4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            # state: (ngf*4) x 8 x 8

            nn.ConvTranspose2d(ngf*4, ngf*2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            # state: (ngf*2) x 16 x 16

            nn.ConvTranspose2d(ngf*2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            # state: ngf x 32 x 32

            nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
            # output: nc x 64 x 64, range [-1, 1]
        )

    def forward(self, x):
        return self.main(x)


In [ ]:
# 4c. Discriminator: 64x64x1 image -> real/fake probability, via convolutions
class Discriminator(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            # input: nc x 64 x 64
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state: ndf x 32 x 32

            nn.Conv2d(ndf, ndf*2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            # state: (ndf*2) x 16 x 16

            nn.Conv2d(ndf*2, ndf*4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            # state: (ndf*4) x 8 x 8

            nn.Conv2d(ndf*4, ndf*8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            # state: (ndf*8) x 4 x 4

            nn.Conv2d(ndf*8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
            # output: 1 x 1 x 1 -> probability real
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


In [ ]:
# 4d. Instantiate models, losses, optimizers
netG = Generator().to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

print(netG)
print(netD)

criterion = nn.BCELoss()

# Fixed noise vector: reused every SAVE_EVERY epochs so we can watch the SAME
# latent points evolve into sharper images over training (a standard DCGAN diagnostic).
fixed_noise = torch.randn(16, NZ, 1, 1, device=device)

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))


In [ ]:
# 4e. Helper to save a 4x4 grid of generator output at a given epoch
def save_grid(epoch, tag="epoch"):
    netG.eval()
    with torch.no_grad():
        fake = netG(fixed_noise).detach().cpu()
    netG.train()
    grid = vutils.make_grid(fake, nrow=4, normalize=True, padding=2)
    npimg = np.transpose(grid.numpy(), (1, 2, 0))

    plt.figure(figsize=(4, 4))
    plt.axis("off")
    plt.title(f"Generated images — {tag} {epoch}")
    plt.imshow(npimg, cmap="gray" if npimg.shape[-1] == 1 else None)
    fname = f"grids/{tag}_{epoch:03d}.png"
    plt.savefig(fname, bbox_inches="tight")
    plt.show()
    plt.close()
    return fname


## 5. Training Loop

Standard DCGAN alternating training:
1. **Discriminator step**: maximize `log(D(real)) + log(1 - D(G(z)))` — one pass on a
   real batch, one pass on a fake batch (detach the generator so its gradients
   aren't affected).
2. **Generator step**: maximize `log(D(G(z)))` (the "non-saturating" trick, rather than
   literally minimizing `log(1 - D(G(z)))`, since it gives much stronger gradients
   early in training when the discriminator easily rejects fakes).

Both G and D losses are logged every batch; a 4×4 grid from the fixed noise vector is
saved every `SAVE_EVERY` epochs.


In [ ]:
# 5. Training
G_losses = []
D_losses = []
D_real_acc = []
D_fake_acc = []
saved_grid_paths = []

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0
    epoch_d_real, epoch_d_fake = 0.0, 0.0

    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # ---------------------
        #  Train Discriminator
        # ---------------------
        netD.zero_grad()

        # real batch
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        output_real = netD(real_imgs)
        lossD_real = criterion(output_real, label)
        lossD_real.backward()

        # fake batch
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_imgs = netG(noise)
        label.fill_(FAKE_LABEL)
        output_fake = netD(fake_imgs.detach())
        lossD_fake = criterion(output_fake, label)
        lossD_fake.backward()

        lossD = lossD_real + lossD_fake
        optimizerD.step()

        # -----------------
        #  Train Generator
        # -----------------
        netG.zero_grad()
        label.fill_(REAL_LABEL)  # generator wants D to think fakes are real
        output = netD(fake_imgs)
        lossG = criterion(output, label)
        lossG.backward()
        optimizerG.step()

        epoch_g_loss += lossG.item()
        epoch_d_loss += lossD.item()
        epoch_d_real += output_real.mean().item()
        epoch_d_fake += output_fake.mean().item()

        G_losses.append(lossG.item())
        D_losses.append(lossD.item())

    n_batches = len(dataloader)
    avg_g, avg_d = epoch_g_loss / n_batches, epoch_d_loss / n_batches
    avg_dr, avg_df = epoch_d_real / n_batches, epoch_d_fake / n_batches
    D_real_acc.append(avg_dr)
    D_fake_acc.append(avg_df)

    elapsed = time.time() - start_time
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"Loss_D: {avg_d:.4f}  Loss_G: {avg_g:.4f}  "
          f"D(real): {avg_dr:.3f}  D(fake): {avg_df:.3f}  "
          f"({elapsed:.0f}s elapsed)")

    if epoch % SAVE_EVERY == 0 or epoch == 1:
        path = save_grid(epoch)
        saved_grid_paths.append(path)

    if epoch % 10 == 0:
        torch.save(netG.state_dict(), f"checkpoints/generator_epoch{epoch}.pth")
        torch.save(netD.state_dict(), f"checkpoints/discriminator_epoch{epoch}.pth")

print("Training complete.")


## 6. Loss Curves

In [ ]:
# 6. Plot generator and discriminator loss over all training iterations
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="Generator", alpha=0.8)
plt.plot(D_losses, label="Discriminator", alpha=0.8)
plt.xlabel("Training iteration (batch)")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("grids/loss_curve.png", bbox_inches="tight")
plt.show()


In [ ]:
# 6b. Per-epoch average D(real) / D(fake) confidence — useful for spotting
# discriminator dominance or mode collapse (values stuck near 0 or 1)
plt.figure(figsize=(10, 4))
plt.plot(range(1, NUM_EPOCHS+1), D_real_acc, label="D(real) avg output")
plt.plot(range(1, NUM_EPOCHS+1), D_fake_acc, label="D(fake) avg output")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="ideal equilibrium (0.5)")
plt.xlabel("Epoch")
plt.ylabel("Discriminator output (probability 'real')")
plt.title("Discriminator confidence on real vs. fake batches")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("grids/discriminator_confidence.png", bbox_inches="tight")
plt.show()


## 7. Generated Image Grids Across Training

In [ ]:
# 7. Display the saved 4x4 grids side by side to see progression over epochs
fig, axes = plt.subplots(1, len(saved_grid_paths), figsize=(4*len(saved_grid_paths), 4))
if len(saved_grid_paths) == 1:
    axes = [axes]

for ax, path in zip(axes, saved_grid_paths):
    img = plt.imread(path)
    ax.imshow(img)
    ax.axis("off")
    epoch_num = path.split("_")[-1].split(".")[0]
    ax.set_title(f"Epoch {int(epoch_num)}")

plt.tight_layout()
plt.savefig("grids/progression_summary.png", bbox_inches="tight")
plt.show()


## 8. Written Observations

*(This section is a template — after you run the notebook, replace the bracketed notes
below with what you actually see in your loss curves and image grids. The expected
pattern for a correctly-training DCGAN on Fashion-MNIST is described here as a guide.)*

**Early epochs (1–5):**
Generated outputs are dominated by random noise, blotchy gray textures, or vague
blob-like shapes with no clear silhouette. The discriminator's loss typically drops
quickly at first because distinguishing real clothing images from pure noise is easy,
while the generator's loss stays high. `[Replace with what you observe in your Epoch 1
and Epoch 5 grids.]`

**Middle epochs (10–20):**
Coarse garment silhouettes start to emerge — rough outlines resembling shirts, bags, or
shoes, though textures remain blurry and edges are soft. The two losses begin to
oscillate rather than monotonically decrease, which is expected in adversarial training:
as the generator improves, the discriminator's job gets harder, and vice versa, so the
losses should hover in a competitive back-and-forth rather than both converging to zero.
`[Replace with what you observe around epoch 10–20 in your run.]`

**Recognisable shapes:**
Clearly identifiable clothing categories (trousers' two-leg silhouette, sneakers'
elongated sole, bags' rectangular body) generally start to appear somewhere in the
`[epoch X]` range in a typical run of this size — note the specific epoch where you
first see this in your own grids.

**Late epochs (25–30) and whether quality plateaus:**
By the final epochs, images should look noticeably sharper and more consistent than the
midpoint, though DCGANs on Fashion-MNIST at this scale (30 epochs, no learning-rate
decay) often plateau in fine detail — outlines are convincing but textures stay somewhat
soft/synthetic. `[State whether your last few grids look meaningfully different from
each other, or whether quality visibly stopped improving.]`

**Mode collapse / instability:**
Watch for: (a) the 4×4 grid showing near-identical images despite different noise
vectors — a sign of mode collapse; (b) D(real) or D(fake) pinned near 1.0 or 0.0 for
many consecutive epochs in the confidence plot — a sign the discriminator has
"won" and stopped providing a useful gradient to the generator; (c) sudden loss spikes
or the generator loss diverging upward — a sign of instability. `[State which, if any,
of these you observed, and at which epoch.]`

**Summary:** `[2-3 sentences summarizing the overall training dynamic you observed —
was it stable, did losses reach a rough equilibrium, did image quality track the loss
curves as expected?]`
